In [0]:
WITH
bronze_stats AS (
    SELECT
        COUNT(*) AS bronze_record_count
    FROM nyc_taxi.bronze.bronze_yellow_trip_2025
),

silver_base AS (
    SELECT
        *,
        COALESCE(
            _dq_reasons,
            CAST(array() AS ARRAY<STRING>)
        ) AS critical_reasons_normalized,
        COALESCE(
            _dq_warning_reasons,
            CAST(array() AS ARRAY<STRING>)
        ) AS warning_reasons_normalized,
        COALESCE(
            _eligibility_reasons,
            CAST(array() AS ARRAY<STRING>)
        ) AS eligibility_reasons_normalized
    FROM nyc_taxi.silver.silver_yellow_trip_2025
),

quarantine_base AS (
    SELECT
        *,
        COALESCE(
            _dq_reasons,
            CAST(array() AS ARRAY<STRING>)
        ) AS critical_reasons_normalized,
        COALESCE(
            _dq_warning_reasons,
            CAST(array() AS ARRAY<STRING>)
        ) AS warning_reasons_normalized,
        COALESCE(
            _eligibility_reasons,
            CAST(array() AS ARRAY<STRING>)
        ) AS eligibility_reasons_normalized
    FROM nyc_taxi.silver.quarantine_yellow_trip_2025
),

silver_stats AS (
    SELECT
        COUNT(*) AS silver_record_count,
        COUNT(DISTINCT record_hash) AS distinct_record_hash_count,

        COUNT_IF(record_hash IS NULL) AS missing_record_hash_count,
        COUNT_IF(NOT (_is_quarantined <=> FALSE))
            AS invalid_silver_quarantine_flag_count,
        COUNT_IF(
            _dq_reasons IS NULL
            OR SIZE(critical_reasons_normalized) <> 0
        ) AS silver_critical_reason_count,
        COUNT_IF(
            NOT (_quality_rule_version <=> '2.0')
        ) AS invalid_quality_rule_version_count,
        COUNT_IF(
            _silver_refresh_timestamp IS NULL
            OR _eligibility_refresh_timestamp IS NULL
        ) AS missing_refresh_timestamp_count,

        COUNT_IF(
            NOT (
                _dq_warning_count
                <=> SIZE(warning_reasons_normalized)
            )
        ) AS warning_count_mismatch_count,
        COUNT_IF(
            NOT (
                _has_dq_warnings
                <=> (SIZE(warning_reasons_normalized) > 0)
            )
        ) AS warning_flag_mismatch_count,
        COUNT_IF(
            NOT (
                _eligibility_restriction_count
                <=> SIZE(eligibility_reasons_normalized)
            )
        ) AS eligibility_count_mismatch_count,
        COUNT_IF(
            _eligibility_status IS NULL
            OR _eligibility_status NOT IN (
                'FULLY_ELIGIBLE',
                'PARTIALLY_ELIGIBLE',
                'REVIEW_REQUIRED'
            )
        ) AS invalid_eligibility_status_count,
        COUNT_IF(
            NOT (
                _requires_data_review
                <=> (_eligibility_status = 'REVIEW_REQUIRED')
            )
        ) AS review_status_mismatch_count,
        COUNT_IF(
            _eligibility_status = 'FULLY_ELIGIBLE'
            AND (
                _eligibility_restriction_count <> 0
                OR NOT (_is_ml_standard_trip_eligible <=> TRUE)
            )
        ) AS fully_eligible_contract_error_count,

        COUNT_IF(
            NOT (_is_trip_volume_metric_eligible <=> TRUE)
        ) AS trip_volume_eligibility_error_count,
        COUNT_IF(
            NOT (_is_recorded_amount_metric_eligible <=> TRUE)
        ) AS recorded_amount_eligibility_error_count,
        COUNT_IF(
            NOT (_is_route_metric_eligible <=> TRUE)
        ) AS route_eligibility_error_count,
        COUNT_IF(
            NOT (
                _is_efficiency_metric_eligible
                <=> (
                    _is_distance_metric_eligible
                    AND _is_duration_metric_eligible
                )
            )
        ) AS efficiency_eligibility_mismatch_count,
        COUNT_IF(
            NOT (
                _is_efficiency_kpi_eligible
                <=> _is_efficiency_metric_eligible
            )
        ) AS efficiency_contract_mismatch_count,
        COUNT_IF(
            NOT (
                _is_passenger_metric_eligible
                <=> (
                    ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'FLEX_FARE_PASSENGER_COUNT_MISSING'
                    ) = FALSE
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'PASSENGER_COUNT_MISSING'
                    ) = FALSE
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'NON_POSITIVE_PASSENGER_COUNT'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_distance_metric_eligible
                <=> (
                    trip_distance > 0
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'ZERO_TRIP_DISTANCE'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_duration_metric_eligible
                <=> (
                    trip_duration_minutes > 0
                    AND _is_long_trip = FALSE
                )
            )
            OR NOT (
                _is_financial_breakdown_metric_eligible
                <=> (
                    ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'FINANCIAL_RECONCILIATION_GAP'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_rate_code_metric_eligible
                <=> (
                    ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'FLEX_FARE_RATE_CODE_MISSING'
                    ) = FALSE
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'RATE_CODE_MISSING_OR_UNMAPPED'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_vendor_metric_eligible
                <=> (
                    ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'VENDOR_MISSING_OR_UNMAPPED'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_payment_type_metric_eligible
                <=> (
                    ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'PAYMENT_TYPE_MISSING_OR_UNMAPPED'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_reported_tip_metric_eligible
                <=> (
                    _is_recorded_amount_metric_eligible
                    AND _is_payment_type_metric_eligible
                    AND payment_type IN (0, 1)
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'TIP_REPORTED_OUTSIDE_SUPPORTED_PAYMENT'
                    ) = FALSE
                )
            )
        ) AS eligibility_warning_rule_mismatch_count,
        COUNT_IF(
            NOT (
                _is_ml_distance_feature_eligible
                <=> _is_distance_metric_eligible
            )
            OR NOT (
                _is_ml_passenger_feature_eligible
                <=> _is_passenger_metric_eligible
            )
            OR NOT (
                _is_ml_categorical_feature_eligible
                <=> (
                    _is_rate_code_metric_eligible
                    AND _is_vendor_metric_eligible
                    AND _is_payment_type_metric_eligible
                )
            )
            OR NOT (
                _is_ml_financial_feature_eligible
                <=> (
                    _is_financial_breakdown_metric_eligible
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'NEGATIVE_TOTAL_AMOUNT'
                    ) = FALSE
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'NEGATIVE_FARE_AMOUNT'
                    ) = FALSE
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'NEGATIVE_TIP_AMOUNT'
                    ) = FALSE
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'NEGATIVE_FINANCIAL_COMPONENT'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_standard_operational_trip_eligible
                <=> (
                    _is_efficiency_metric_eligible
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'NEGATIVE_TOTAL_AMOUNT'
                    ) = FALSE
                    AND ARRAY_CONTAINS(
                        warning_reasons_normalized,
                        'DUPLICATE_GROUP_CANONICAL_RECORD'
                    ) = FALSE
                )
            )
            OR NOT (
                _is_ml_standard_trip_eligible
                <=> (
                    _is_standard_operational_trip_eligible
                    AND _is_passenger_metric_eligible
                    AND _is_ml_financial_feature_eligible
                    AND _is_ml_categorical_feature_eligible
                )
            )
        ) AS ml_eligibility_mismatch_count,

        COUNT_IF(passenger_count < 0)
            AS negative_passenger_in_silver_count,
        COUNT_IF(
            NOT (
                ARRAY_CONTAINS(
                    warning_reasons_normalized,
                    'ZERO_TRIP_DISTANCE'
                )
                <=> (trip_distance = 0)
            )
        ) AS zero_distance_warning_mismatch_count,
        COUNT_IF(
            passenger_count IS NULL
            AND payment_type = 0
            AND NOT ARRAY_CONTAINS(
                warning_reasons_normalized,
                'FLEX_FARE_PASSENGER_COUNT_MISSING'
            )
        ) AS flex_fare_passenger_warning_mismatch_count,
        COUNT_IF(
            passenger_count IS NULL
            AND (payment_type IS NULL OR payment_type <> 0)
            AND NOT ARRAY_CONTAINS(
                warning_reasons_normalized,
                'PASSENGER_COUNT_MISSING'
            )
        ) AS unexpected_passenger_warning_mismatch_count,
        COUNT_IF(
            passenger_count <= 0
            AND NOT ARRAY_CONTAINS(
                warning_reasons_normalized,
                'NON_POSITIVE_PASSENGER_COUNT'
            )
        ) AS non_positive_passenger_warning_mismatch_count,
        COUNT_IF(
            NOT (
                ARRAY_CONTAINS(
                    warning_reasons_normalized,
                    'FINANCIAL_RECONCILIATION_GAP'
                )
                <=> _is_financially_unreconciled
            )
            OR NOT (
                ARRAY_CONTAINS(
                    warning_reasons_normalized,
                    'NEGATIVE_TOTAL_AMOUNT'
                )
                <=> _is_negative_total_amount
            )
            OR NOT (
                ARRAY_CONTAINS(
                    warning_reasons_normalized,
                    'NEGATIVE_FARE_AMOUNT'
                )
                <=> _is_negative_fare_amount
            )
            OR NOT (
                ARRAY_CONTAINS(
                    warning_reasons_normalized,
                    'NEGATIVE_TIP_AMOUNT'
                )
                <=> _is_negative_tip_amount
            )
            OR NOT (
                ARRAY_CONTAINS(
                    warning_reasons_normalized,
                    'NEGATIVE_FINANCIAL_COMPONENT'
                )
                <=> _has_negative_financial_component
            )
        ) AS financial_warning_mismatch_count,
        COUNT_IF(
            NOT (
                _requires_gold_date_extension
                <=> ARRAY_CONTAINS(
                    warning_reasons_normalized,
                    'CROSS_YEAR_TRIP'
                )
            )
        ) AS gold_date_extension_mismatch_count,

        COUNT_IF(
            ARRAY_CONTAINS(
                warning_reasons_normalized,
                'FLEX_FARE_PASSENGER_COUNT_MISSING'
            )
        ) AS flex_fare_missing_passenger_count,
        COUNT_IF(
            ARRAY_CONTAINS(
                warning_reasons_normalized,
                'PASSENGER_COUNT_MISSING'
            )
        ) AS unexpected_missing_passenger_count,
        COUNT_IF(
            ARRAY_CONTAINS(
                warning_reasons_normalized,
                'ZERO_TRIP_DISTANCE'
            )
        ) AS zero_distance_record_count,
        COUNT_IF(
            ARRAY_CONTAINS(
                warning_reasons_normalized,
                'FINANCIAL_RECONCILIATION_GAP'
            )
        ) AS financial_reconciliation_gap_count,
        COUNT_IF(
            ARRAY_CONTAINS(
                warning_reasons_normalized,
                'NEGATIVE_TOTAL_AMOUNT'
            )
        ) AS negative_total_amount_count,
        COUNT_IF(_requires_gold_date_extension)
            AS gold_date_extension_required_count,
        COUNT_IF(_eligibility_status = 'REVIEW_REQUIRED')
            AS review_required_count,
        COUNT_IF(_eligibility_status = 'PARTIALLY_ELIGIBLE')
            AS partially_eligible_count
    FROM silver_base
),

quarantine_stats AS (
    SELECT
        COUNT(*) AS quarantine_record_count,
        COUNT_IF(record_hash IS NULL)
            AS missing_record_hash_count,
        COUNT_IF(NOT (_is_quarantined <=> TRUE))
            AS invalid_quarantine_flag_count,
        COUNT_IF(
            _dq_reasons IS NULL
            OR SIZE(critical_reasons_normalized) = 0
        ) AS missing_critical_reason_count,
        COUNT_IF(
            NOT (_quality_rule_version <=> '2.0')
        ) AS invalid_quality_rule_version_count,
        COUNT_IF(
            _silver_refresh_timestamp IS NULL
            OR _eligibility_refresh_timestamp IS NULL
        ) AS missing_refresh_timestamp_count,
        COUNT_IF(
            NOT (_requires_data_review <=> TRUE)
            OR NOT (_eligibility_status <=> 'REVIEW_REQUIRED')
        ) AS invalid_review_status_count,
        COUNT_IF(
            NOT (_is_trip_volume_metric_eligible <=> FALSE)
            OR NOT (_is_recorded_amount_metric_eligible <=> FALSE)
            OR NOT (_is_route_metric_eligible <=> FALSE)
            OR NOT (_is_ml_standard_trip_eligible <=> FALSE)
        ) AS invalid_quarantine_eligibility_count,
        COUNT_IF(
            NOT (
                _eligibility_restriction_count
                <=> SIZE(eligibility_reasons_normalized)
            )
            OR NOT ARRAY_CONTAINS(
                eligibility_reasons_normalized,
                'CRITICAL_QUALITY_VIOLATION'
            )
        ) AS invalid_quarantine_reason_contract_count,
        COUNT_IF(
            NOT (
                _dq_warning_count
                <=> SIZE(warning_reasons_normalized)
            )
            OR NOT (
                _has_dq_warnings
                <=> (SIZE(warning_reasons_normalized) > 0)
            )
        ) AS quarantine_warning_contract_mismatch_count,
        COUNT_IF(
            NOT (
                ARRAY_CONTAINS(
                    critical_reasons_normalized,
                    'NEGATIVE_PASSENGER_COUNT'
                )
                <=> COALESCE(passenger_count < 0, FALSE)
            )
        ) AS negative_passenger_reason_mismatch_count
    FROM quarantine_base
),

validation_results AS (
    SELECT
        10 AS validation_order,
        'VOLUME' AS validation_group,
        'source_record_conservation' AS validation_name,
        'BLOCKER' AS severity,
        CASE
            WHEN b.bronze_record_count =
                s.silver_record_count + q.quarantine_record_count
            THEN 'PASS'
            ELSE 'FAIL'
        END AS status,
        ABS(
            b.bronze_record_count
            - s.silver_record_count
            - q.quarantine_record_count
        ) AS exception_record_count,
        b.bronze_record_count AS evaluated_record_count,
        CONCAT(
            'bronze=', b.bronze_record_count,
            '; silver=', s.silver_record_count,
            '; quarantine=', q.quarantine_record_count
        ) AS details
    FROM bronze_stats b
    CROSS JOIN silver_stats s
    CROSS JOIN quarantine_stats q

    UNION ALL

    SELECT
        20, 'SILVER_CONTRACT', 'silver_is_populated', 'BLOCKER',
        CASE WHEN silver_record_count > 0 THEN 'PASS' ELSE 'FAIL' END,
        CASE WHEN silver_record_count > 0 THEN 0 ELSE 1 END,
        silver_record_count,
        CONCAT('silver_records=', silver_record_count)
    FROM silver_stats

    UNION ALL

    SELECT
        30, 'SILVER_CONTRACT', 'record_hash_is_unique', 'BLOCKER',
        CASE
            WHEN silver_record_count = distinct_record_hash_count
            THEN 'PASS' ELSE 'FAIL'
        END,
        silver_record_count - distinct_record_hash_count,
        silver_record_count,
        CONCAT(
            'rows=', silver_record_count,
            '; distinct_hashes=', distinct_record_hash_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        40, 'SILVER_CONTRACT', 'record_hash_is_present', 'BLOCKER',
        CASE WHEN missing_record_hash_count = 0 THEN 'PASS' ELSE 'FAIL' END,
        missing_record_hash_count, silver_record_count,
        'record_hash must be populated in every published Silver record'
    FROM silver_stats

    UNION ALL

    SELECT
        50, 'SILVER_CONTRACT', 'no_critical_record_in_silver', 'BLOCKER',
        CASE
            WHEN invalid_silver_quarantine_flag_count = 0
             AND silver_critical_reason_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        invalid_silver_quarantine_flag_count + silver_critical_reason_count,
        silver_record_count,
        CONCAT(
            'invalid_flags=', invalid_silver_quarantine_flag_count,
            '; records_with_critical_reasons=', silver_critical_reason_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        60, 'METADATA_CONTRACT', 'silver_rule_version_and_refresh', 'BLOCKER',
        CASE
            WHEN invalid_quality_rule_version_count = 0
             AND missing_refresh_timestamp_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        invalid_quality_rule_version_count + missing_refresh_timestamp_count,
        silver_record_count,
        CONCAT(
            'invalid_rule_versions=', invalid_quality_rule_version_count,
            '; missing_refresh_timestamps=', missing_refresh_timestamp_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        70, 'WARNING_CONTRACT', 'warning_array_count_and_flag', 'BLOCKER',
        CASE
            WHEN warning_count_mismatch_count = 0
             AND warning_flag_mismatch_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        warning_count_mismatch_count + warning_flag_mismatch_count,
        silver_record_count,
        CONCAT(
            'count_mismatches=', warning_count_mismatch_count,
            '; flag_mismatches=', warning_flag_mismatch_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        80, 'WARNING_CONTRACT', 'warning_business_rules', 'BLOCKER',
        CASE
            WHEN zero_distance_warning_mismatch_count = 0
             AND flex_fare_passenger_warning_mismatch_count = 0
             AND unexpected_passenger_warning_mismatch_count = 0
             AND non_positive_passenger_warning_mismatch_count = 0
             AND financial_warning_mismatch_count = 0
             AND gold_date_extension_mismatch_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        zero_distance_warning_mismatch_count
            + flex_fare_passenger_warning_mismatch_count
            + unexpected_passenger_warning_mismatch_count
            + non_positive_passenger_warning_mismatch_count
            + financial_warning_mismatch_count
            + gold_date_extension_mismatch_count,
        silver_record_count,
        CONCAT(
            'zero_distance=', zero_distance_warning_mismatch_count,
            '; flex_passenger=', flex_fare_passenger_warning_mismatch_count,
            '; unexpected_passenger=', unexpected_passenger_warning_mismatch_count,
            '; non_positive_passenger=', non_positive_passenger_warning_mismatch_count,
            '; financial=', financial_warning_mismatch_count,
            '; gold_date=', gold_date_extension_mismatch_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        90, 'ELIGIBILITY_CONTRACT', 'eligibility_reason_count_and_status',
        'BLOCKER',
        CASE
            WHEN eligibility_count_mismatch_count = 0
             AND invalid_eligibility_status_count = 0
             AND review_status_mismatch_count = 0
             AND fully_eligible_contract_error_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        eligibility_count_mismatch_count
            + invalid_eligibility_status_count
            + review_status_mismatch_count
            + fully_eligible_contract_error_count,
        silver_record_count,
        CONCAT(
            'reason_count=', eligibility_count_mismatch_count,
            '; invalid_status=', invalid_eligibility_status_count,
            '; review_status=', review_status_mismatch_count,
            '; fully_eligible=', fully_eligible_contract_error_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        100, 'ELIGIBILITY_CONTRACT', 'mandatory_metric_eligibility', 'BLOCKER',
        CASE
            WHEN trip_volume_eligibility_error_count = 0
             AND recorded_amount_eligibility_error_count = 0
             AND route_eligibility_error_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        trip_volume_eligibility_error_count
            + recorded_amount_eligibility_error_count
            + route_eligibility_error_count,
        silver_record_count,
        CONCAT(
            'trip_volume=', trip_volume_eligibility_error_count,
            '; recorded_amount=', recorded_amount_eligibility_error_count,
            '; route=', route_eligibility_error_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        110, 'ELIGIBILITY_CONTRACT', 'derived_eligibility_consistency', 'BLOCKER',
        CASE
            WHEN efficiency_eligibility_mismatch_count = 0
             AND efficiency_contract_mismatch_count = 0
             AND eligibility_warning_rule_mismatch_count = 0
             AND ml_eligibility_mismatch_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        efficiency_eligibility_mismatch_count
            + efficiency_contract_mismatch_count
            + eligibility_warning_rule_mismatch_count
            + ml_eligibility_mismatch_count,
        silver_record_count,
        CONCAT(
            'efficiency_formula=', efficiency_eligibility_mismatch_count,
            '; efficiency_kpi=', efficiency_contract_mismatch_count,
            '; warning_rules=', eligibility_warning_rule_mismatch_count,
            '; ml_formula=', ml_eligibility_mismatch_count
        )
    FROM silver_stats

    UNION ALL

    SELECT
        120, 'BUSINESS_RULE', 'negative_passenger_is_quarantined', 'BLOCKER',
        CASE
            WHEN negative_passenger_in_silver_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        negative_passenger_in_silver_count,
        silver_record_count,
        'Negative passenger_count must not be published in the valid Silver table'
    FROM silver_stats

    UNION ALL

    SELECT
        130, 'QUARANTINE_CONTRACT', 'quarantine_has_critical_reasons', 'BLOCKER',
        CASE
            WHEN invalid_quarantine_flag_count = 0
             AND missing_critical_reason_count = 0
             AND missing_record_hash_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        invalid_quarantine_flag_count
            + missing_critical_reason_count
            + missing_record_hash_count,
        quarantine_record_count,
        CONCAT(
            'invalid_flags=', invalid_quarantine_flag_count,
            '; missing_reasons=', missing_critical_reason_count,
            '; missing_hashes=', missing_record_hash_count
        )
    FROM quarantine_stats

    UNION ALL

    SELECT
        140, 'QUARANTINE_CONTRACT', 'quarantine_eligibility_and_metadata',
        'BLOCKER',
        CASE
            WHEN invalid_quality_rule_version_count = 0
             AND missing_refresh_timestamp_count = 0
             AND invalid_review_status_count = 0
             AND invalid_quarantine_eligibility_count = 0
             AND invalid_quarantine_reason_contract_count = 0
             AND quarantine_warning_contract_mismatch_count = 0
             AND negative_passenger_reason_mismatch_count = 0
            THEN 'PASS' ELSE 'FAIL'
        END,
        invalid_quality_rule_version_count
            + missing_refresh_timestamp_count
            + invalid_review_status_count
            + invalid_quarantine_eligibility_count
            + invalid_quarantine_reason_contract_count
            + quarantine_warning_contract_mismatch_count
            + negative_passenger_reason_mismatch_count,
        quarantine_record_count,
        CONCAT(
            'metadata=',
                invalid_quality_rule_version_count + missing_refresh_timestamp_count,
            '; review_status=', invalid_review_status_count,
            '; eligibility=', invalid_quarantine_eligibility_count,
            '; reasons=', invalid_quarantine_reason_contract_count,
            '; warnings=', quarantine_warning_contract_mismatch_count,
            '; negative_passenger=', negative_passenger_reason_mismatch_count
        )
    FROM quarantine_stats

    UNION ALL

    SELECT
        200, 'MONITORING', 'unexpected_missing_passenger', 'WARNING',
        CASE
            WHEN unexpected_missing_passenger_count = 0
            THEN 'PASS' ELSE 'WARN'
        END,
        unexpected_missing_passenger_count,
        silver_record_count,
        'Unexpected missing passenger_count; exclude from passenger metrics and ML features'
    FROM silver_stats

    UNION ALL

    SELECT
        210, 'MONITORING', 'flex_fare_missing_passenger', 'INFO',
        'INFO', flex_fare_missing_passenger_count, silver_record_count,
        'Known Flex Fare limitation; trip volume and amount remain eligible'
    FROM silver_stats

    UNION ALL

    SELECT
        220, 'MONITORING', 'zero_trip_distance', 'WARNING',
        CASE WHEN zero_distance_record_count = 0 THEN 'PASS' ELSE 'WARN' END,
        zero_distance_record_count, silver_record_count,
        'Records remain in Silver but are excluded from distance and efficiency metrics'
    FROM silver_stats

    UNION ALL

    SELECT
        230, 'MONITORING', 'financial_reconciliation_gap', 'WARNING',
        CASE
            WHEN financial_reconciliation_gap_count = 0
            THEN 'PASS' ELSE 'WARN'
        END,
        financial_reconciliation_gap_count, silver_record_count,
        'Records require review and are excluded from financial breakdown metrics'
    FROM silver_stats

    UNION ALL

    SELECT
        240, 'MONITORING', 'negative_total_amount', 'WARNING',
        CASE WHEN negative_total_amount_count = 0 THEN 'PASS' ELSE 'WARN' END,
        negative_total_amount_count, silver_record_count,
        'Review as reversal/adjustment; do not automatically discard signed transactions'
    FROM silver_stats

    UNION ALL

    SELECT
        250, 'MONITORING', 'gold_date_extension_required', 'WARNING',
        CASE
            WHEN gold_date_extension_required_count = 0
            THEN 'PASS' ELSE 'WARN'
        END,
        gold_date_extension_required_count, silver_record_count,
        'Extend dim_date through the maximum dropoff date before rebuilding Gold'
    FROM silver_stats

    UNION ALL

    SELECT
        260, 'MONITORING', 'records_requiring_review', 'WARNING',
        CASE WHEN review_required_count = 0 THEN 'PASS' ELSE 'WARN' END,
        review_required_count, silver_record_count,
        CONCAT(
            'review_required=', review_required_count,
            '; partially_eligible=', partially_eligible_count
        )
    FROM silver_stats
)

SELECT
    validation_group,
    validation_name,
    severity,
    status,
    exception_record_count,
    evaluated_record_count,
    ROUND(
        CASE
            WHEN evaluated_record_count = 0 THEN 0.0
            ELSE exception_record_count * 100.0 / evaluated_record_count
        END,
        6
    ) AS exception_rate_pct,
    details
FROM validation_results
ORDER BY validation_order;